In [3]:
"""
Graph Classification using GCN, GAT, and GraphSAGE (NumPy implementation)
==========================================================================
Dataset:
  - features.csv  : Node features (2000 nodes × 16 features)
  - edges.csv     : Undirected edges (8000 edges)
  - labels.csv    : Node-level binary labels (used for graph-level task)

Since the dataset is a single graph (node classification setting),
we convert it to a graph classification task by:
  - Splitting nodes into 10 subgraphs (communities by index)
  - Each subgraph's label = majority label of its nodes
  - Graph-level representation = mean pooling of node embeddings
"""

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, accuracy_score, classification_report

# ─────────────────────────────────────────────
# 1.  Load Data
# ─────────────────────────────────────────────
print("=" * 60)
print("  Graph Classification: GCN | GAT | GraphSAGE")
print("=" * 60)

features_df = pd.read_csv("features.csv")
labels_df   = pd.read_csv("labels.csv")
edges_df    = pd.read_csv("edges.csv")

# Sort by Vertex to ensure alignment
features_df.columns = features_df.columns.str.strip()
labels_df.columns = labels_df.columns.str.strip()
edges_df.columns = edges_df.columns.str.strip()
features_df = features_df.sort_values("Vertex").reset_index(drop=True)
labels_df   = labels_df.sort_values("Vertex").reset_index(drop=True)

N = len(features_df)                        # 2000 nodes
feature_cols = [c for c in features_df.columns if c != "Vertex"]
X = features_df[feature_cols].values.astype(np.float32)   # (2000, 16)
y = labels_df["label"].values.astype(np.int32)             # (2000,)

# Normalize features
X = (X - X.mean(0)) / (X.std(0) + 1e-8)

src = edges_df["Vertex 1"].values
dst = edges_df["Vertex 2"].values

# ─────────────────────────────────────────────
# 2.  Build Adjacency & Degree Matrices
# ─────────────────────────────────────────────
# Symmetric adjacency (add self-loops for GCN)
adj = np.zeros((N, N), dtype=np.float32)
for s, d in zip(src, dst):
    adj[s, d] = 1.0
    adj[d, s] = 1.0

# Adjacency with self-loops
adj_hat = adj + np.eye(N, dtype=np.float32)

# Degree matrix D^{-1/2}
deg = adj_hat.sum(axis=1)
D_inv_sqrt = np.diag(1.0 / np.sqrt(deg + 1e-8))
A_norm = D_inv_sqrt @ adj_hat @ D_inv_sqrt   # Ã = D^-½ Â D^-½

# For GraphSAGE: D^{-1} (row-normalised adjacency without self-loops)
deg2 = adj.sum(axis=1, keepdims=True) + 1e-8
A_sage = adj / deg2   # row-normalised

# For GAT: neighbour list
neighbours = {i: list(np.where(adj[i] > 0)[0]) for i in range(N)}

# ─────────────────────────────────────────────
# 3.  Graph Splitting for Classification
# ─────────────────────────────────────────────
NUM_GRAPHS   = 200
NODES_PER_G  = N // NUM_GRAPHS   # 10 nodes each

def make_graph_dataset(node_embeddings, y, num_graphs, nodes_per_g):
    """Mean-pool node embeddings into graph-level vectors."""
    Xg, yg = [], []
    for g in range(num_graphs):
        idx = range(g * nodes_per_g, (g + 1) * nodes_per_g)
        Xg.append(node_embeddings[idx].mean(0))
        yg.append(int(y[idx].mean() >= 0.5))    # majority label
    return np.array(Xg, dtype=np.float32), np.array(yg, dtype=np.int32)

# ─────────────────────────────────────────────
# 4.  Helper Functions
# ─────────────────────────────────────────────
def relu(x):
    return np.maximum(0, x)

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -50, 50)))

def softmax(x):
    e = np.exp(x - x.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

def cross_entropy_loss(probs, labels):
    n = len(labels)
    log_p = np.log(probs[np.arange(n), labels] + 1e-10)
    return -log_p.mean()

def he_init(fan_in, fan_out):
    return np.random.randn(fan_in, fan_out).astype(np.float32) * np.sqrt(2.0 / fan_in)

def train_classifier(Xg_tr, yg_tr, Xg_te, yg_te,
                     hidden=64, lr=0.01, epochs=200, model_name=""):
    """Simple 2-layer MLP classifier on graph embeddings."""
    np.random.seed(42)
    in_dim = Xg_tr.shape[1]
    W1 = he_init(in_dim, hidden)
    b1 = np.zeros((1, hidden), dtype=np.float32)
    W2 = he_init(hidden, 2)
    b2 = np.zeros((1, 2), dtype=np.float32)

    for ep in range(epochs):
        # Forward
        h1 = relu(Xg_tr @ W1 + b1)
        logits = h1 @ W2 + b2
        probs  = softmax(logits)
        loss   = cross_entropy_loss(probs, yg_tr)

        # Backward
        dL = probs.copy()
        dL[np.arange(len(yg_tr)), yg_tr] -= 1
        dL /= len(yg_tr)

        dW2 = h1.T @ dL
        db2 = dL.sum(0, keepdims=True)
        dh1 = dL @ W2.T * (h1 > 0)
        dW1 = Xg_tr.T @ dh1
        db1 = dh1.sum(0, keepdims=True)

        W1 -= lr * dW1; b1 -= lr * db1
        W2 -= lr * dW2; b2 -= lr * db2

        if (ep + 1) % 50 == 0:
            print(f"  [{model_name}] Epoch {ep+1:3d}/{epochs}  Loss: {loss:.4f}")

    # Predict on test
    h1_te = relu(Xg_te @ W1 + b1)
    preds = (softmax(h1_te @ W2 + b2)).argmax(axis=1)
    return preds

# ─────────────────────────────────────────────
# 5.  GCN  –  Graph Convolutional Network
# ─────────────────────────────────────────────
print("\n" + "─" * 60)
print("  MODEL 1: GCN (Graph Convolutional Network)")
print("─" * 60)

np.random.seed(0)
in_dim, h_dim, out_dim = X.shape[1], 64, 32

W_gcn1 = he_init(in_dim, h_dim)
W_gcn2 = he_init(h_dim, out_dim)

# Two-layer GCN message passing:  H^(l+1) = σ( Ã H^(l) W^(l) )
H1_gcn = relu(A_norm @ X   @ W_gcn1)   # (N, 64)
H2_gcn = relu(A_norm @ H1_gcn @ W_gcn2) # (N, 32)

Xg_gcn, yg = make_graph_dataset(H2_gcn, y, NUM_GRAPHS, NODES_PER_G)
Xg_tr, Xg_te, yg_tr, yg_te = train_test_split(
    Xg_gcn, yg, test_size=0.2, random_state=42, stratify=yg)

preds_gcn = train_classifier(Xg_tr, yg_tr, Xg_te, yg_te, model_name="GCN")

print("\n  ── GCN Results ──")
print(f"  Accuracy  : {accuracy_score(yg_te, preds_gcn):.4f}")
print(f"  Precision : {precision_score(yg_te, preds_gcn, zero_division=0):.4f}")
print(f"  Recall    : {recall_score(yg_te, preds_gcn, zero_division=0):.4f}")
print("\n  Classification Report:")
print(classification_report(yg_te, preds_gcn, target_names=["Class 0", "Class 1"]))

# ─────────────────────────────────────────────
# 6.  GAT  –  Graph Attention Network
# ─────────────────────────────────────────────
print("─" * 60)
print("  MODEL 2: GAT (Graph Attention Network)")
print("─" * 60)

np.random.seed(1)
NUM_HEADS = 4
HEAD_DIM  = 16   # per-head dimension  → output = 64

# Weight matrices per head (layer 1)
W_gat = [he_init(in_dim, HEAD_DIM) for _ in range(NUM_HEADS)]
# Attention weight vectors [a1 | a2] per head
a_src = [np.random.randn(HEAD_DIM, 1).astype(np.float32) * 0.01 for _ in range(NUM_HEADS)]
a_dst = [np.random.randn(HEAD_DIM, 1).astype(np.float32) * 0.01 for _ in range(NUM_HEADS)]

def gat_layer(X, W_list, a_src_list, a_dst_list, neighbours, leaky=0.2):
    """Single GAT layer with multi-head attention (mean aggregation)."""
    num_heads = len(W_list)
    N = X.shape[0]
    head_outputs = []

    for k in range(num_heads):
        Wh = X @ W_list[k]                     # (N, HEAD_DIM)
        e_src = (Wh @ a_src_list[k]).flatten()  # (N,)
        e_dst = (Wh @ a_dst_list[k]).flatten()  # (N,)

        H_out = np.zeros_like(Wh)
        for i in range(N):
            nb = neighbours[i]
            if len(nb) == 0:
                H_out[i] = Wh[i]
                continue
            # Attention scores (LeakyReLU)
            scores = np.array([e_src[i] + e_dst[j] for j in nb], dtype=np.float32)
            scores = np.where(scores > 0, scores, leaky * scores)  # LeakyReLU
            # Softmax
            scores -= scores.max()
            exp_s  = np.exp(scores)
            alpha  = exp_s / (exp_s.sum() + 1e-10)
            H_out[i] = (alpha[:, None] * Wh[nb]).sum(0)

        head_outputs.append(H_out)

    # Concatenate heads
    return relu(np.concatenate(head_outputs, axis=1))  # (N, HEAD_DIM * NUM_HEADS)

print("  Computing GAT layer 1 (multi-head attention) ...")
H1_gat = gat_layer(X, W_gat, a_src, a_dst, neighbours)    # (N, 64)

# Layer 2: single head, output 32
W_gat2 = [he_init(H1_gat.shape[1], 32)]
a_src2 = [np.random.randn(32, 1).astype(np.float32) * 0.01]
a_dst2 = [np.random.randn(32, 1).astype(np.float32) * 0.01]
print("  Computing GAT layer 2 ...")
H2_gat = gat_layer(H1_gat, W_gat2, a_src2, a_dst2, neighbours)  # (N, 32)

Xg_gat, _ = make_graph_dataset(H2_gat, y, NUM_GRAPHS, NODES_PER_G)
Xg_tr, Xg_te, yg_tr, yg_te = train_test_split(
    Xg_gat, yg, test_size=0.2, random_state=42, stratify=yg)

preds_gat = train_classifier(Xg_tr, yg_tr, Xg_te, yg_te, model_name="GAT")

print("\n  ── GAT Results ──")
print(f"  Accuracy  : {accuracy_score(yg_te, preds_gat):.4f}")
print(f"  Precision : {precision_score(yg_te, preds_gat, zero_division=0):.4f}")
print(f"  Recall    : {recall_score(yg_te, preds_gat, zero_division=0):.4f}")
print("\n  Classification Report:")
print(classification_report(yg_te, preds_gat, target_names=["Class 0", "Class 1"]))

# ─────────────────────────────────────────────
# 7.  GraphSAGE
# ─────────────────────────────────────────────
print("─" * 60)
print("  MODEL 3: GraphSAGE")
print("─" * 60)

np.random.seed(2)
# GraphSAGE:  h_v = σ( W · CONCAT(h_v, MEAN({h_u : u ∈ N(v)})) )
W_sage1 = he_init(in_dim * 2, 64)   # concat self + mean-neighbour
W_sage2 = he_init(64 * 2,    32)

def sage_layer(X, A_sage, W):
    """One GraphSAGE layer: concat self-embedding and mean-neighbour."""
    neigh_agg = A_sage @ X                          # mean of neighbours (N, in_dim)
    concat    = np.concatenate([X, neigh_agg], axis=1)
    H         = relu(concat @ W)
    # L2-normalise rows
    norms = np.linalg.norm(H, axis=1, keepdims=True) + 1e-8
    return H / norms

H1_sage = sage_layer(X,       A_sage, W_sage1)   # (N, 64)
H2_sage = sage_layer(H1_sage, A_sage, W_sage2)   # (N, 32)

Xg_sage, _ = make_graph_dataset(H2_sage, y, NUM_GRAPHS, NODES_PER_G)
Xg_tr, Xg_te, yg_tr, yg_te = train_test_split(
    Xg_sage, yg, test_size=0.2, random_state=42, stratify=yg)

preds_sage = train_classifier(Xg_tr, yg_tr, Xg_te, yg_te, model_name="GraphSAGE")

print("\n  ── GraphSAGE Results ──")
print(f"  Accuracy  : {accuracy_score(yg_te, preds_sage):.4f}")
print(f"  Precision : {precision_score(yg_te, preds_sage, zero_division=0):.4f}")
print(f"  Recall    : {recall_score(yg_te, preds_sage, zero_division=0):.4f}")
print("\n  Classification Report:")
print(classification_report(yg_te, preds_sage, target_names=["Class 0", "Class 1"]))

# ─────────────────────────────────────────────
# 8.  Summary Table
# ─────────────────────────────────────────────
print("=" * 60)
print("  FINAL SUMMARY")
print("=" * 60)
print(f"  {'Model':<14}  {'Accuracy':>10}  {'Precision':>10}  {'Recall':>10}")
print(f"  {'─'*14}  {'─'*10}  {'─'*10}  {'─'*10}")
for name, preds in [("GCN", preds_gcn), ("GAT", preds_gat), ("GraphSAGE", preds_sage)]:
    acc  = accuracy_score(yg_te, preds)
    prec = precision_score(yg_te, preds, zero_division=0)
    rec  = recall_score(yg_te, preds, zero_division=0)
    print(f"  {name:<14}  {acc:>10.4f}  {prec:>10.4f}  {rec:>10.4f}")
print("=" * 60)

  Graph Classification: GCN | GAT | GraphSAGE

────────────────────────────────────────────────────────────
  MODEL 1: GCN (Graph Convolutional Network)
────────────────────────────────────────────────────────────
  [GCN] Epoch  50/200  Loss: 0.2892
  [GCN] Epoch 100/200  Loss: 0.1556
  [GCN] Epoch 150/200  Loss: 0.1001
  [GCN] Epoch 200/200  Loss: 0.0713

  ── GCN Results ──
  Accuracy  : 1.0000
  Precision : 1.0000
  Recall    : 1.0000

  Classification Report:
              precision    recall  f1-score   support

     Class 0       1.00      1.00      1.00        20
     Class 1       1.00      1.00      1.00        20

    accuracy                           1.00        40
   macro avg       1.00      1.00      1.00        40
weighted avg       1.00      1.00      1.00        40

────────────────────────────────────────────────────────────
  MODEL 2: GAT (Graph Attention Network)
────────────────────────────────────────────────────────────
  Computing GAT layer 1 (multi-head attent